# Data Cleaning & Structural Validation

**Objective:** Clean messy customer data with pandas and validate its structure.

**Dataset note:** The provided raw CSV in this package is synthetic demonstration data because no official sample dataset was attached. Replace it with the dataset from your assignment if one is specified.

## Cleaning rules
- Standardize column headers and trim whitespace.
- Convert placeholder strings to missing values.
- Remove exact duplicate records.
- Normalize city and membership capitalization.
- Parse dates; invalid dates become missing.
- Convert numeric columns and flag impossible ages.
- Fill missing name/membership with `Unknown`, purchase amount with the median, and invalid order counts with 0 for this demonstration.
- Leave unknown email, age and signup date missing rather than fabricating values.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

input_file = Path("messy_sample_dataset.csv")
output_file = Path("cleaned_customer_dataset.csv")
missing_tokens = {"", "unknown", "n/a", "na", "not available", "null", "none"}

df = pd.read_csv(input_file, dtype=str)
rows_before = len(df)
df.columns = ["_".join(str(c).strip().lower().replace("-", " ").split()) for c in df.columns]
df = df.replace(r"^\\s*$", np.nan, regex=True)
for col in df.columns:
    df[col] = df[col].map(lambda x: np.nan if isinstance(x, str) and x.strip().lower() in missing_tokens else x)

duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

df["city"] = df["city"].str.title()
df["membership"] = df["membership"].str.title()
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce", dayfirst=False).dt.strftime("%Y-%m-%d")
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df.loc[(df["age"] < 0) | (df["age"] > 100), "age"] = np.nan
df["age"] = df["age"].round().astype("Int64")
df["purchase_amount"] = pd.to_numeric(df["purchase_amount"], errors="coerce")
median_amount = df["purchase_amount"].median()
df["purchase_amount"] = df["purchase_amount"].fillna(median_amount).round(2)
df["orders"] = pd.to_numeric(df["orders"], errors="coerce").fillna(0).astype("Int64")
df["customer_name"] = df["customer_name"].fillna("Unknown")
df["membership"] = df["membership"].fillna("Unknown")
df = df.dropna(subset=["customer_id"]).copy()

print("Rows before cleaning:", rows_before)
print("Exact duplicates removed:", duplicates_removed)
print("Rows after cleaning:", len(df))
print("\\nMissing values after cleaning:")
print(df.isna().sum())
df.head()

## Structural validation

In [ ]:
assert df.columns.is_unique, "Duplicate column headers found"
assert df["customer_id"].is_unique, "Customer IDs are not unique; investigate before using data"
assert pd.api.types.is_numeric_dtype(df["age"])
assert pd.api.types.is_numeric_dtype(df["purchase_amount"])
assert pd.api.types.is_numeric_dtype(df["orders"])
assert df["age"].dropna().between(0, 100).all()
print("Structural checks passed.")
print(df.dtypes)

In [ ]:
df.to_csv(output_file, index=False)
print("Cleaned CSV saved to:", output_file.resolve())
df.head(10)